<a href="https://colab.research.google.com/github/sam098761212-glitch/Leeseungsoo-AI_CLASS/blob/main/diabetes_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, accuracy_score

tf.keras.utils.set_random_seed(42)
if not os.path.exists("diabetes.csv"):
    from google.colab import files
    files.upload()

df = pd.read_csv("diabetes.csv")
zero_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

def preprocess(x_train, x_test):
    cols = [c for c in zero_missing if c in x_train.columns]
    x_train = x_train.copy()
    x_test = x_test.copy()
    x_train[cols] = x_train[cols].replace(0, np.nan)
    x_test[cols] = x_test[cols].replace(0, np.nan)
    imputer = SimpleImputer(strategy="median")
    x_train = imputer.fit_transform(x_train)
    x_test = imputer.transform(x_test)
    scaler = StandardScaler()
    return scaler.fit_transform(x_train).astype("float32"), scaler.transform(x_test).astype("float32")

In [2]:
# 당뇨병 딥러닝 회귀: Glucose(혈당) 수치 예측
reg_df = df[df["Glucose"] > 0].copy()
X_reg = reg_df.drop(columns=["Glucose", "Outcome"])
y_reg = reg_df["Glucose"].to_numpy(dtype="float32")
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
Xr_train, Xr_test = preprocess(Xr_train, Xr_test)
regression_model = Sequential([
    layers.Input(shape=(Xr_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])
regression_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
regression_model.fit(
    Xr_train, yr_train, epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=1
)
reg_pred = regression_model.predict(Xr_test, verbose=0).ravel()
print("Diabetes 회귀 MAE:", mean_absolute_error(yr_test, reg_pred))


Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 15970.4756 - mae: 122.5682 - val_loss: 14387.7334 - val_mae: 116.4087
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 15775.7549 - mae: 121.7851 - val_loss: 14177.9990 - val_mae: 115.5285
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 15530.0781 - mae: 120.7861 - val_loss: 13900.2852 - val_mae: 114.3490
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 15192.1582 - mae: 119.3999 - val_loss: 13518.3926 - val_mae: 112.7055
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14720.0117 - mae: 117.4349 - val_loss: 12988.3936 - val_mae: 110.3823
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14069.6973 - mae: 114.6688 - val_loss: 12273.0986 - val_mae: 107.1601
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 13198.1816 - mae: 110.8444 - val_loss: 11336.0693 - val_mae: 102.7741
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 12073.6680 - mae: 105.6844 - val_loss

In [3]:
# 당뇨병 딥러닝 분류: Outcome(0=음성, 1=양성) 예측
X_cls = df.drop(columns=["Outcome"])
y_cls = df["Outcome"].to_numpy(dtype="int32")
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
Xc_train, Xc_test = preprocess(Xc_train, Xc_test)
classification_model = Sequential([
    layers.Input(shape=(Xc_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
classification_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
classification_model.fit(
    Xc_train, yc_train, epochs=100, batch_size=32, validation_split=0.2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=1
)
cls_pred = (classification_model.predict(Xc_test, verbose=0).ravel() >= 0.5).astype(int)
print("Diabetes 분류 정확도:", accuracy_score(yc_test, cls_pred))


Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5316 - loss: 0.7005 - val_accuracy: 0.6911 - val_loss: 0.5906
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7251 - loss: 0.5878 - val_accuracy: 0.7724 - val_loss: 0.5274
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7434 - loss: 0.5342 - val_accuracy: 0.7886 - val_loss: 0.4851
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7658 - loss: 0.4999 - val_accuracy: 0.7967 - val_loss: 0.4568
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7780 - loss: 0.4773 - val_accuracy: 0.7967 - val_loss: 0.4400
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7780 - loss: 0.4622 - val_accuracy: 0.7805 - val_loss: 0.4295
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7862 - loss: 0.4515 - val_accuracy: 0.7805 - val_loss: 0.4221
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7923 - loss: 0.4435 - val_accuracy: 0.7